In [0]:
spark.sql(
    "DROP TABLE IF EXISTS workspace.gold.fact_earthquake"
)

In [0]:
spark.sql("""
CREATE TABLE IF NOT EXISTS workspace.gold.fact_earthquake (
    earthquake_id STRING,
    
    date_key BIGINT,
    location_key BIGINT,
    magnitude_key BIGINT,
    event_type_key BIGINT,
    alert_key BIGINT,
    network_key BIGINT,

    magnitude DOUBLE,
    depth_km DOUBLE,
    felt_count BIGINT,
    cdi DOUBLE,
    mmi DOUBLE,
    significance BIGINT,
    station_count BIGINT,
    distance_min DOUBLE,
    rms DOUBLE,
    azimuthal_gap DOUBLE,
    
    tsunami_flag BOOLEAN,
    status STRING
)
""")

In [0]:
from pyspark.sql import functions as F

earthquake = spark.table(
    "workspace.silver.usgs_earthquakes"
)

In [0]:
dim_time = spark.table(
    "workspace.gold.dim_time"
)

dim_location = spark.table(
    "workspace.gold.dim_location"
)

dim_magnitude = spark.table(
    "workspace.gold.dim_magnitude"
)

dim_event_type = spark.table(
    "workspace.gold.dim_event_type"
)

dim_alert = spark.table(
    "workspace.gold.dim_alert"
)

dim_network = spark.table(
    "workspace.gold.dim_network"
)

In [0]:
earthquake = earthquake.withColumn(
    "date_key",
    F.date_format(
        F.to_date("event_timestamp"),
        "yyyyMMdd"
    ).cast("bigint")
)

In [0]:
earthquake = earthquake.withColumn(
    "magnitude_category",
    F.when(
        F.col("magnitude") < 2,
        "Menor a 2"
    )
    .when(
        F.col("magnitude") < 4,
        "2 a menor de 4"
    )
    .when(
        F.col("magnitude") < 5,
        "4 a menor de 5"
    )
    .when(
        F.col("magnitude") < 6,
        "5 a menor de 6"
    )
    .when(
        F.col("magnitude") < 7,
        "6 a menor de 7"
    )
    .otherwise(
        "7 o más"
    )
)

In [0]:
fact = (
    earthquake.alias("e")

    .join(
        dim_time.alias("dt"),
        F.col("e.date_key") ==
        F.col("dt.date_key"),
        "left"
    )

    .join(
        dim_location.alias("dl"),
        (
            F.col("e.place") ==
            F.col("dl.place")
        )
        &
        (
            F.col("e.latitude") ==
            F.col("dl.latitude")
        )
        &
        (
            F.col("e.longitude") ==
            F.col("dl.longitude")
        ),
        "left"
    )

    .join(
        dim_magnitude.alias("dm"),
        (
            F.col("e.magnitude_type") ==
            F.col("dm.magnitude_type")
        )
        &
        (
            F.col("e.magnitude_category") ==
            F.col("dm.magnitude_category")
        ),
        "left"
    )

    .join(
        dim_event_type.alias("det"),
        F.col("e.event_type") ==
        F.col("det.event_type"),
        "left"
    )

    .join(
        dim_alert.alias("da"),
        F.coalesce(
            F.col("e.alert"),
            F.lit("No alert")
        ) ==
        F.col("da.alert_level"),
        "left"
    )

    .join(
        dim_network.alias("dn"),
        F.coalesce(
            F.col("e.network"),
            F.lit("UNKNOWN")
        ) ==
        F.col("dn.network_code"),
        "left"
    )
)

In [0]:
fact = fact.select(
    F.col("e.earthquake_id"),

    F.col("dt.date_key"),
    F.col("dl.location_key"),
    F.col("dm.magnitude_key"),
    F.col("det.event_type_key"),
    F.col("da.alert_key"),
    F.col("dn.network_key"),

    F.col("e.magnitude"),
    F.col("e.depth").alias("depth_km"),
    F.col("e.felt").alias("felt_count"),
    F.col("e.cdi"),
    F.col("e.mmi"),
    F.col("e.significance"),
    F.col("e.station_count"),
    F.col("e.distance_min"),
    F.col("e.rms"),
    F.col("e.azimuthal_gap"),

    F.col("e.tsunami_flag"),
    F.col("e.status")
)

In [0]:
fact = fact.dropDuplicates(
    ["earthquake_id"]
)

In [0]:
fact.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .saveAsTable(
        "workspace.gold.fact_earthquake"
    )